# Reaction Video Editor — Colab Version

**What this is:** A Python pipeline combining both repo variants (root original + `diffrent variant`) for processing reaction videos inside Google Colab.

**What it does:**
- Splits 3840×1080 OBS side-by-side recordings into camera / content halves.
- Composes reaction layout (camera top-left rounded, content bottom-right rounded, blurred background 50%/op 40%).
- Mixes 2-track audio with compressor / limiter / sidechain ducking.
- Auto-cuts silent/repeated reaction segments for seamless flow.
- Transcribes intro/outro via Whisper to fix pauses/repeats.
- Applies face retouch (smooth, teeth, nose, eyes) with MediaPipe tracking.
- Exports MP4 and WebM directly to Google Drive.

**Where computing happens:** Inside this Colab VM (CPU; enable GPU runtime if you want). Large 3 GB files stream through ffmpeg.

**WebM to YouTube:** Yes — YouTube fully accepts VP9/WebM. We export both MP4 and WebM.

**Drive input/output:** Mount with `drive.mount()`, point `ReactionVideoProcessor` to your video, set `output_dir` to Drive.

In [ ]:
# 1) Install dependencies (run once per session)
!pip install -q numpy opencv-python mediapipe openai-whisper ffmpeg-python moviepy pydub ipywidgets

## 2) Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3) Import processor and configure paths

In [ ]:
# Clone the repo (fresh copy every session) and import the pipeline
!rm -rf /content/VideoEditorTool
!git clone -q https://github.com/RailRGO/VideoEditorTool.git /content/VideoEditorTool
!ls /content/VideoEditorTool/colab_version/   # expect video_processor.py, compose.py, layouts.py, editor_gui.py

import sys
sys.path.insert(0, '/content/VideoEditorTool/colab_version')
from video_processor import ReactionVideoProcessor
print('import OK')

In [ ]:
input_video = '/content/drive/MyDrive/raw/recording_3840.mp4'   # <-- change to your file
output_dir  = '/content/drive/MyDrive/reaction_output'          # <-- change if you like

proc = ReactionVideoProcessor(input_video, output_dir=output_dir)

## 3b) Visual editor — see every tweak before you render

Run the cell below to open the editor GUI. **Every slider updates a live preview** rendered by the exact same compositor as the final file (WYSIWYG), including an overlap warning if the camera covers the content.

Workflow:
1. **1 · Preview** — scrub time, pick the scene (reaction / solo / card / lead-in).
2. **2 · Layout** — preset + camera/content rects, shapes, radius, borders, background blur.
3. **3 · Retouch** — face smoothing / teeth / eyes with live preview.
4. **4 · Cuts** — intro/outro, silence auto-cut, YouTube claim ranges.
5. **5 · Audio** — compressor / limiter / ducking + audible 10s sample.
6. **6 · Render** — render a **12s sample at the playhead first** (video + mixed audio), watch it, and only then hit **RENDER FULL VIDEO**.

Your tuned layout lives on `proc.layout`, so the Patreon/YouTube cells below pick it up automatically. You can also save it to Drive (`layout.json`) and reload it next session.

In [ ]:
from editor_gui import launch_editor

editor = launch_editor(proc)   # <- the whole editor UI appears below

## 3c) Full web app — the original editor experience, in a browser tab

Prefer the old app over cell widgets? This starts a web server **on the Colab VM** and gives you a public link. Open it in a new tab:

- Smooth playback with **draggable camera/content boxes**, timeline, transport — like the original app.
- Every heavy job (pixel-exact stills, audio/video samples, full renders) runs on the Colab VM against your real file.
- The link dies with the session; re-run the cell for a fresh one. Don't share it publicly.

**How the link is made:** the cell tries several no-account tunnels in order — Cloudflare quick tunnel (QUIC, then HTTP/2), then ssh→localhost.run on port 443 and port 22 — and **only prints a URL it has verified works from the outside**. Colab blocks different outbound paths at different times, so if the first one can't get out the next is tried automatically. If *all* fail, it prints exactly what failed (download / tunnel start / reachability) and what to do next — re-run the cell, restart the session, or use the in-cell editor from cell 3b.

First launch transcodes a small proxy in the background (one-time, cached in your output folder on Drive); you can keep editing with exact stills meanwhile, or wait ~2–5 min for smooth playback.

In [ ]:
from webapp.server import launch_webapp

# Prints a verified https:// URL (trycloudflare.com or lhr.life) — open it in a new browser tab.
# The URL is reachability-checked before it's shown (takes ~10–90s).
# Safe to re-run: it keeps the server/state and just makes a fresh tunnel.
# If every tunnel option fails, read the printed diagnostics — they name the
# exact step that failed (download / tunnel start / reachability).
# To fully stop everything: from webapp.server import stop_webapp; stop_webapp()
web = launch_webapp(proc)

## 3d) Hosted editor — the full React UI on a weak PC

Your machine doesn't need to process anything. Host the editor UI once (see `render.yaml` — one click on Render, free tier is fine), then:

1. Run this cell — it starts the same backend as 3c (or reuses it) and prints the tunnel URL.
2. Open your hosted site, switch the engine to **Colab**, paste the URL, press **Connect**.
3. Pick the source file (any video in the same Drive folder), edit on the proxy preview, hit **Render on Colab**.

The timeline, layout, audio, retouch and cloak settings you see are sent to this notebook as a project file; the render runs here against the full-resolution original, and the finished MP4 downloads through the same tunnel. Closing the browser tab mid-render is safe — reconnect later and the download is still waiting in the Render tab.

In [ ]:
# 3d) Backend for the HOSTED editor (weak-PC workflow).
# Reuses the server from 3c when it's already up, otherwise starts it
# now (server + verified tunnel, ~10-90s on first launch).
try:
    web
except NameError:
    from webapp.server import launch_webapp
    web = launch_webapp(proc)

print()
print('Paste this URL into the hosted editor (engine: Colab):')
print(f"  {web['public'] or web['local']}")

## 4) Patreon version — full uncut + intro/outro cleaned
- Intro / outro kept full camera (safe, no copyrighted content).
- Reaction composed with preset layout.
- Whispers intro to detect pauses / repeats for manual edit.

In [ ]:
# Patreon: full uncut reaction, intro/outro in full-cam.
# Uses your tuned layout from the editor if you opened it.
layout = editor.layout if 'editor' in globals() else None

proc.run_patron_version(
    intro_range=(0, 45),
    outro_range=(1250, 1290),
    retouch=False,
    fix_intro=True,
    layout=layout,
)

## 5) YouTube version — reaction with alterations + auto-cuts
- Middle reaction gets silence/repeat cuts (auto_cut=True).
- Camera retouched (retouch=True) — tracking rebuilds mask every frame.
- Custom cuts can be passed: custom_cuts=[(t1,t2), ...]

In [ ]:
# YouTube: cut-down reaction (silence drops + claims + retouch).
# Drops/claims/layout come from the editor when it's open.
if 'editor' in globals():
    proc.cuts_cfg['claims'] = list(editor.claims)
    _drops = list(editor.drops)
    _layout = editor.layout
else:
    _drops, _layout = [], None

proc.run_youtube_version(
    auto_cut=('editor' not in globals()),  # editor already found silences
    retouch=True,
    intro_range=(0, 45),
    outro_range=(1250, 1290),
    custom_cuts=_drops,
    layout=_layout,
)

## 5b) No-widget fallback: stills + sample clip from plain code

If widgets ever misbehave, these two lines give you the same WYSIWYG proof without any GUI:

In [ ]:
# Still preview at 90 seconds, reaction scene:
proc.show_preview(t=90, mode='body')

# ...or a real 12-second clip (composed video + mixed audio) you can play:
# outs = proc.render_sample(t_center=90, seconds=12, mode='body')
# from IPython.display import Video; Video(filename=outs['mp4'], embed=True, width=720)

## 6) Check output files in Drive
Files written to `/content/drive/MyDrive/reaction_output/` (or your `output_dir`):
- `patreon_final.mp4` / `.webm`
- `youtube_final.mp4` / `.webm`
- `intro_transcript.json` (Whisper result)

List them with:
```python
!ls -lh /content/drive/MyDrive/reaction_output/
```

In [ ]:
!ls -lh /content/drive/MyDrive/reaction_output/ 2>/dev/null || echo 'Output folder not found yet — run pipeline above first.'

---
**Notes**
- If session dies, re-run cells 1→3; outputs in Drive survive.
- Very long videos: split at source (e.g., 0-10 min chunks) and concatenate with `ffmpeg -i p1 -i p2 -filter_complex concat`.
- Face retouch is CPU-heavy; for 20 min videos it may take 10–20 min. Skip retouch if you prefer CapCut for that step.